# Day 20 — Coverage & Phase 1 Week 4 Wrap

> ⚠️ **Why this matters.** "I have tests" is not enough. **Coverage** measures *which lines* your tests execute. 100% coverage doesn't mean correct, but <50% almost always means broken-without-knowing-it.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakkzz/prince-curriculum/blob/main/phase-1-python-cli/lessons/20-coverage.ipynb)

## What you'll do today

- [ ] You can install and run `coverage` with pytest
- [ ] You can read a coverage report and find untested code
- [ ] english-helper has ≥80% coverage on core modules
- [ ] You've added a CI workflow that runs tests + coverage

## 1. Install + first report

In [ ]:
# uv add --dev coverage pytest-cov
#
# Then:
# uv run pytest --cov=src/english_helper --cov-report=term-missing

**Sample output:**

```
tests/ ........              [100%]

---------- coverage ----------
Name                    Stmts   Miss  Cover   Missing
-------------------------------------------------
src/.../word.py           20      0   100%
src/.../storage.py        45      3    93%   23-25
src/.../api.py            30     12    60%   18-29
-------------------------------------------------
TOTAL                     95     15    84%
```

`Missing` shows the line numbers NOT covered. **Read it. Fix the gaps that matter.**

## 2. HTML reports

In [ ]:
# uv run pytest --cov=src/english_helper --cov-report=html
# open htmlcov/index.html

The HTML report color-codes each source line: green=covered, red=missed. Click into any file to see exactly what your tests touched.

**Use this once a week** to spot untested branches you didn't realize were there.

## 3. What to chase (and not)

**Worth chasing:**
- Untested **business logic** (the scheduling, the quiz grading)
- Untested **error paths** — these are exactly the ones that fail in production
- Untested **state transitions** (load → modify → save)

**NOT worth chasing:**
- Defensive `if __name__ == '__main__':` blocks
- Trivial `__repr__` (covered by other tests anyway)
- Hard-to-mock infrastructure (full retry-with-backoff timing)

> 💡 **Coverage rule:** target 80%, hit 90% on critical modules, don't bend over backwards for 100%. The marginal cost rises sharply above 90%.

## 4. Configure coverage in pyproject.toml

In [ ]:
# pyproject.toml
[tool.coverage.run]
source = ['src/english_helper']
branch = true   # also check branch coverage (both arms of if)

[tool.coverage.report]
exclude_lines = [
    'pragma: no cover',
    'if __name__ == .__main__.:',
    'raise NotImplementedError',
]
fail_under = 80   # CI fails if total coverage drops below

Now just `uv run pytest --cov` uses these settings.

`fail_under = 80` means: when running in CI, a PR that drops coverage below 80% fails. Forces discipline.

## 5. CI — GitHub Actions

Create `.github/workflows/test.yml`:

```yaml
name: tests
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: astral-sh/setup-uv@v3
      - run: uv python install
      - run: uv sync --frozen
      - run: uv run ruff check .
      - run: uv run mypy src
      - run: uv run pytest --cov=src/english_helper --cov-report=term-missing
```

Push it. Open a PR. Watch CI run.

**Green badge on the README is the ultimate "this works" signal.**

## End-of-day — bring english-helper to 80% coverage

> 🎯 **The final Week 4 mission:**

1. Run coverage. Note your current %.
2. Read the `Missing` lines.
3. Add tests targeting the most important uncovered code.
4. Repeat until you're at 80%+ overall, and 90%+ on `word.py` + `storage.py` + `dictionary.py`.
5. Add CI: workflow runs on every PR.
6. Add coverage badge to README.

Acceptance: `uv run pytest --cov` shows ≥80% total. CI green on a PR.

## Phase 1 Week 4 retrospective

Things you can now do that you couldn't a week ago:

- Write tests for new features before writing them (TDD)
- Mock external services so tests are fast and deterministic
- Share setup with fixtures
- Measure what your tests actually cover
- Wire up CI to enforce all of this on every PR

These are **senior engineer skills**. Most CS graduates don't have them. You do.

## Connect to the project

> 🎯 **End of Week 4. Next week (Days 21-25):** packaging + shipping. You'll polish english-helper, build pomo (the Phase 1 capstone), and ship both as installable CLI tools.

**Quiz:** [20-coverage-quiz.ipynb](20-coverage-quiz.ipynb)